# Part 3b — Embedding Training: txt+img | time-independent
**Bach et al. (2025) Appendix G**

- Modality : Text (RoBERTa) + Image (BEiT) + Tabular (SAINT)
- Mode      : Time-independent (no lag)
- Outputs   : 6 zip files in `predictions/txtimg/time_indipendent/`

**A100 GPU required. ~45-90 min.**

## ① Mount Drive

In [ ]:
# Local mode - no Google Drive needed
print('Local mode')

## ② Setup Environment

Installs pytorch-widedeep to an isolated folder and patches the gensim
dependency conflict. No restart needed.

In [ ]:
import sys, os, types

# ── Dependencies (install locally before running) ────────────────────────────
# pip install pytorch-widedeep==1.7.0 scipy==1.13.1 torchmetrics
print('Dependencies: pytorch-widedeep, scipy, torchmetrics (install locally)')

# ── Inject fake gensim to block all C extension conflicts ─────────────────────
# pytorch-widedeep imports gensim for its text utilities only
# We do not use those, SAINT only needs the tabular components
fake_utils = types.ModuleType('gensim.utils')
fake_utils.tokenize = lambda text, *a, **kw: text.lower().split()
for mod_name in [
    'gensim', 'gensim.utils', 'gensim.parsing',
    'gensim.parsing.preprocessing', 'gensim.corpora',
    'gensim.matutils', 'gensim.interfaces',
    'gensim.models', 'gensim.similarities',
]:
    sys.modules[mod_name] = types.ModuleType(mod_name)
sys.modules['gensim.utils'] = fake_utils
print('gensim conflict bypassed')

In [ ]:
import torch
import os
import torch.nn as nn
import numpy as np
import pandas as pd
from datetime import datetime
from transformers import AutoTokenizer, AutoModel, BeitModel, BeitImageProcessor
from pytorch_widedeep.models.tabular.transformers.saint import SAINT
from pytorch_widedeep.preprocessing.tab_preprocessor import TabPreprocessor

print(f'torch            : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
    print(f'VRAM             : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device           : {DEVICE}')
print('✅ All imports successful')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Switch to A100 GPU runtime.')

## ③ Config

In [ ]:
import os
# Detect project root
from pathlib import Path as _P
_cwd = _P.cwd()
_root = _cwd
for _ in range(5):
    if (_root / 'data').is_dir() and (_root / 'code').is_dir():
        break
    _root = _root.parent
else:
    raise RuntimeError('Cannot find project root')
ROOT = str(_root) + '/'
DATA_DIR  = ROOT + 'data/'
IMG_DIR   = DATA_DIR + 'images/'
PRED_DIR  = DATA_DIR + 'predictions/'
SPLIT_DIR = DATA_DIR + 'amzn_shoes_monthly_diffs_ffill_fixed_splits/'

BATCH_SIZE   = 32
EPOCHS_LEVEL = 25
EPOCHS_DIFF  = 15
LR           = 2e-5
EMB_DIMS     = [128, 256]
MOD          = 4
WINDOW       = 28
MAX_PERIODS  = 53
SEED         = 42
ENCODER_DIM  = 768

TEXT_MODEL_NAME  = 'cardiffnlp/twitter-roberta-base'
IMAGE_MODEL_NAME = 'microsoft/beit-base-patch16-224'

TAB_COLS = [
    'RATING',
    'REVIEW_COUNT',
    'New Offer Count: Current',
    'Count of retrieved live offers: New, FBA',
    'Count of retrieved live offers: New, FBM',
    'Lightning Deals: Upcoming Deal',
    'Buy Box: Is FBA',
]
TAB_CONTINUOUS = TAB_COLS  # all treated as continuous

for subdir in [
    'txt/time_indipendent', 'txt/lag1',
    'txtimg/time_indipendent', 'txtimg/lag1',
]:
    os.makedirs(PRED_DIR + subdir, exist_ok=True)

TODAY = datetime.now().strftime('%Y-%m-%d')
torch.manual_seed(SEED)
print(f'Config ready | TODAY={TODAY}')
print(f'Pred dir: {PRED_DIR}')

## ④ Load and Prepare Panel Data

In [ ]:
df_train_raw = pd.read_parquet(SPLIT_DIR + 'train-00000-of-00001.parquet')
df_val_raw   = pd.read_parquet(SPLIT_DIR + 'validation-00000-of-00001.parquet')

def prepare_df(df):
    df = df[df['window'] == float(WINDOW)].copy()
    df = df.dropna(subset=['SALES_RANK', 'PRICE'])
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['ASIN', 'date']).reset_index(drop=True)
    df['date_t'] = df['date'].astype('category').cat.codes
    df = df[df['date_t'] <= MAX_PERIODS]
    df = df[df['date_t'] % MOD == 0]
    for col in TAB_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(float)
    return df.reset_index(drop=True)

df_train = prepare_df(df_train_raw)
df_val   = prepare_df(df_val_raw)

print(f'Train : {len(df_train):,} rows | {df_train["ASIN"].nunique():,} ASINs | {df_train["date_t"].nunique()} periods')
print(f'Val   : {len(df_val):,} rows | {df_val["ASIN"].nunique():,} ASINs | {df_val["date_t"].nunique()} periods')

## ⑤ First Differences and Lag-1 Features

In [ ]:
def add_diffs(df):
    df = df.sort_values(['ASIN', 'date']).copy()
    df['DELTA_SALES_RANK'] = df.groupby('ASIN')['SALES_RANK'].diff()
    df['DELTA_PRICE']      = df.groupby('ASIN')['PRICE'].diff()
    return df.dropna(subset=['DELTA_SALES_RANK', 'DELTA_PRICE']).reset_index(drop=True)

def add_lag1(df):
    df = df.sort_values(['ASIN', 'date']).copy()
    df['SALES_RANK_lag1'] = df.groupby('ASIN')['SALES_RANK'].shift(1)
    df['PRICE_lag1']      = df.groupby('ASIN')['PRICE'].shift(1)
    return df.dropna(subset=['SALES_RANK_lag1', 'PRICE_lag1']).reset_index(drop=True)

df_train_level     = df_train.copy()
df_val_level       = df_val.copy()
df_train_diff      = add_diffs(df_train)
df_val_diff        = add_diffs(df_val)
df_train_lag1      = add_lag1(df_train_level)
df_val_lag1        = add_lag1(df_val_level)
df_train_diff_lag1 = add_lag1(df_train_diff)
df_val_diff_lag1   = add_lag1(df_val_diff)

print(f'Level     train/val : {len(df_train_level):,} / {len(df_val_level):,}')
print(f'Diff      train/val : {len(df_train_diff):,} / {len(df_val_diff):,}')
print(f'Lag1      train/val : {len(df_train_lag1):,} / {len(df_val_lag1):,}')
print(f'Diff+Lag1 train/val : {len(df_train_diff_lag1):,} / {len(df_val_diff_lag1):,}')

## ⑥ Load Encoders and Fit Tabular Preprocessor

In [ ]:
print('Loading text tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
print(f'  ✅ {TEXT_MODEL_NAME}')

print('Loading image processor...')
image_processor = BeitImageProcessor.from_pretrained(IMAGE_MODEL_NAME)
print(f'  ✅ {IMAGE_MODEL_NAME}')

print('Fitting tabular preprocessor...')
all_tab = pd.concat([df_train_level, df_val_level])[TAB_COLS].copy()
tab_preprocessor = TabPreprocessor(
    continuous_cols = TAB_COLS,
    cat_embed_cols  = None,
    scale           = True,
)
tab_preprocessor.fit(all_tab)

# pytorch-widedeep 1.7.0 uses cat_embed_input from column_idx directly
_cat_embed_input = getattr(tab_preprocessor, 'cat_embed_input', None)

_saint_test = SAINT(
    column_idx      = tab_preprocessor.column_idx,
    cat_embed_input = _cat_embed_input,
    continuous_cols = TAB_COLS,
    input_dim       = 32,
    n_heads         = 4,
    n_blocks        = 2,
)

_saint_test.eval()
_dummy = torch.tensor(
    tab_preprocessor.transform(
        pd.DataFrame(np.zeros((2, len(TAB_COLS))), columns=TAB_COLS)
    ), dtype=torch.float32
)
with torch.no_grad():
    SAINT_OUT_DIM = _saint_test(_dummy).shape[-1]
del _saint_test

print(f'  ✅ TabPreprocessor fitted | SAINT output dim: {SAINT_OUT_DIM}')

## ⑦ Model Architecture (Appendix G, Figure 18)

In [ ]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, context):
        q  = query.unsqueeze(1)
        kv = context.unsqueeze(1)
        out, _ = self.attn(q, kv, kv)
        return self.norm(query + self.dropout(out.squeeze(1)))


class FullDemandModel(nn.Module):
    """
    Full multimodal embedding model — Bach et al. (2025) Appendix G.
    Three encoders fused via all-to-all Cross Attention Blocks.
    All parameters fine-tuned end-to-end.
    """
    def __init__(self, emb_dim, txt_only, use_lag, saint_out_dim, enc_dim=768):
        super().__init__()
        self.txt_only = txt_only
        self.use_lag  = use_lag

        # Pretrained encoders
        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME)
        if not txt_only:
            self.image_encoder = BeitModel.from_pretrained(IMAGE_MODEL_NAME)
            self.saint_encoder = SAINT(
                column_idx      = tab_preprocessor.column_idx,
                cat_embed_input = getattr(tab_preprocessor, 'cat_embed_input', None),
                continuous_cols = TAB_COLS,
                input_dim       = 32,
                n_heads         = 4,
                n_blocks        = 2,
            )
            self.tab_proj = (
                nn.Linear(saint_out_dim, enc_dim)
                if saint_out_dim != enc_dim else nn.Identity()
            )

        # Layer Norms
        self.norm_txt = nn.LayerNorm(enc_dim)
        if not txt_only:
            self.norm_img = nn.LayerNorm(enc_dim)
            self.norm_tab = nn.LayerNorm(enc_dim)
            # Cross Attention Blocks — all-to-all
            self.cab_txt  = CrossAttentionBlock(enc_dim)
            self.cab_img  = CrossAttentionBlock(enc_dim)
            self.cab_tab  = CrossAttentionBlock(enc_dim)
            fusion_dim = enc_dim * 3
        else:
            fusion_dim = enc_dim

        lag_dim = 2 if use_lag else 0

        # FC projection to embedding E
        self.projection = nn.Sequential(
            nn.Linear(fusion_dim + lag_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, emb_dim),
        )
        # Two output heads for DoubleML
        self.head_q = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU(), nn.Linear(64, 1))
        self.head_p = nn.Sequential(nn.Linear(emb_dim, 64), nn.GELU(), nn.Linear(64, 1))

    def forward(self, input_ids, attention_mask,
                pixel_values=None, tab_tensor=None, lag_feats=None):
        e_txt = self.norm_txt(
            self.text_encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state[:, 0, :]
        )
        if not self.txt_only:
            e_img = self.norm_img(
                self.image_encoder(pixel_values=pixel_values).last_hidden_state[:, 0, :]
            )
            e_tab = self.norm_tab(
                self.tab_proj(self.saint_encoder(tab_tensor))
            )
            e_txt_f = self.cab_txt(e_txt, (e_img + e_tab) / 2)
            e_img_f = self.cab_img(e_img, (e_txt + e_tab) / 2)
            e_tab_f = self.cab_tab(e_tab, (e_txt + e_img) / 2)
            fused   = torch.cat([e_txt_f, e_img_f, e_tab_f], dim=-1)
        else:
            fused = e_txt

        if self.use_lag and lag_feats is not None:
            fused = torch.cat([fused, lag_feats], dim=-1)

        E     = self.projection(fused)
        q_hat = self.head_q(E).squeeze(-1)
        p_hat = self.head_p(E).squeeze(-1)
        return E, q_hat, p_hat


print('✅ Architecture defined')

## ⑧ Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage

class ShoesDataset(Dataset):
    def __init__(self, df, target_q, target_p, use_lag=False):
        self.df       = df.reset_index(drop=True)
        self.target_q = target_q
        self.target_p = target_p
        self.use_lag  = use_lag
        self._tok_cache = {}
        self._img_cache = {}

    def _tok(self, asin, text):
        if asin not in self._tok_cache:
            enc = tokenizer(str(text), padding='max_length', truncation=True,
                            max_length=128, return_tensors='pt')
            self._tok_cache[asin] = {
                'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
            }
        return self._tok_cache[asin]

    def _img(self, asin):
        if asin not in self._img_cache:
            path = IMG_DIR + f'{asin}.jpg'
            img  = PILImage.open(path).convert('RGB') if os.path.exists(path)                    else PILImage.new('RGB', (224, 224), (128, 128, 128))
            enc  = image_processor(images=img, return_tensors='pt')
            self._img_cache[asin] = enc['pixel_values'].squeeze(0)
        return self._img_cache[asin]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        asin = row['ASIN']
        tok  = self._tok(asin, row['text'])
        tab  = tab_preprocessor.transform(
            pd.DataFrame([row[TAB_COLS].values.astype(float)], columns=TAB_COLS)
        ).squeeze(0)
        item = {
            'input_ids':      tok['input_ids'],
            'attention_mask': tok['attention_mask'],
            'pixel_values':   self._img(asin),
            'tab_tensor':     torch.tensor(tab, dtype=torch.float32),
            'q': torch.tensor(float(row[self.target_q]), dtype=torch.float32),
            'p': torch.tensor(float(row[self.target_p]),  dtype=torch.float32),
        }
        if self.use_lag:
            item['lag_feats'] = torch.tensor(
                [float(row['SALES_RANK_lag1']), float(row['PRICE_lag1'])],
                dtype=torch.float32
            )
        return item


def make_loader(df, shuffle, target_q='SALES_RANK', target_p='PRICE', use_lag=False):
    ds = ShoesDataset(df, target_q, target_p, use_lag)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=4, pin_memory=True)

print('✅ Dataset defined')

## ⑨ Training Utilities

In [ ]:
import torch.optim as optim

def train_one_epoch(model, loader, optimizer, scaler, txt_only):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE) if not txt_only else None
        tab  = batch['tab_tensor'].to(DEVICE)   if not txt_only else None
        lag  = batch['lag_feats'].to(DEVICE) if 'lag_feats' in batch else None
        q, p = batch['q'].to(DEVICE), batch['p'].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            _, q_hat, p_hat = model(ids, mask, pix, tab, lag)
            loss = nn.MSELoss()(q_hat, q) + nn.MSELoss()(p_hat, p)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item(); n += 1
    return total / n


@torch.no_grad()
def eval_loss(model, loader, txt_only):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE) if not txt_only else None
        tab  = batch['tab_tensor'].to(DEVICE)   if not txt_only else None
        lag  = batch['lag_feats'].to(DEVICE) if 'lag_feats' in batch else None
        q, p = batch['q'].to(DEVICE), batch['p'].to(DEVICE)
        with torch.amp.autocast('cuda'):
            _, q_hat, p_hat = model(ids, mask, pix, tab, lag)
            loss = nn.MSELoss()(q_hat, q) + nn.MSELoss()(p_hat, p)
        total += loss.item(); n += 1
    return total / n


@torch.no_grad()
def extract_and_save(model, df, txt_only, save_path, emb_dim):
    model.eval()
    target_q = 'DELTA_SALES_RANK' if 'DELTA_SALES_RANK' in df.columns else 'SALES_RANK'
    target_p = 'DELTA_PRICE'      if 'DELTA_PRICE'      in df.columns else 'PRICE'
    ds     = ShoesDataset(df, target_q, target_p, model.use_lag)
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4)
    all_emb, all_q, all_p = [], [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE) if not txt_only else None
        tab  = batch['tab_tensor'].to(DEVICE)   if not txt_only else None
        lag  = batch['lag_feats'].to(DEVICE) if 'lag_feats' in batch else None
        with torch.amp.autocast('cuda'):
            emb, q_hat, p_hat = model(ids, mask, pix, tab, lag)
        all_emb.append(emb.cpu().float().numpy())
        all_q.append(q_hat.cpu().float().numpy())
        all_p.append(p_hat.cpu().float().numpy())
    emb_arr  = np.concatenate(all_emb)
    q_arr    = np.concatenate(all_q)
    p_arr    = np.concatenate(all_p)
    df_r     = df.reset_index(drop=True)
    df_out   = pd.DataFrame(emb_arr, columns=[str(i) for i in range(emb_dim)])
    df_out.insert(0, 'pred_ml_m', p_arr)
    df_out.insert(0, 'pred_ml_l', q_arr)
    df_out.insert(0, 'time',  df_r['date'].dt.strftime('%Y-%m-%d'))
    df_out.insert(0, 'index', df_r['ASIN'])
    csv_name = os.path.basename(save_path).replace('.zip', '.csv')
    df_out.to_csv(save_path,
                  compression=dict(method='zip', archive_name=csv_name),
                  index=False)
    mb = os.path.getsize(save_path) / 1e6
    print(f'    → {os.path.basename(save_path)}  ({mb:.1f} MB, {len(df_out):,} rows)')


print('✅ Training utilities defined')

## ⑩ Main Training Loop — All 4 Variants

Order:
1. txt  + time_independent
2. txt  + lag1
3. txtimg + time_independent
4. txtimg + lag1

In [ ]:
VARIANTS = [
    (False, False, 'time_independent', 'txtimg/time_indipendent'),
]

for txt_only, use_lag, lag_type, subfolder in VARIANTS:
    modality = 'txt' if txt_only else 'txtimg'
    print(f'\n' + '='*65)
    print(f'VARIANT : {modality} | {lag_type}')
    print('='*65)

    tr_lv = df_train_lag1      if use_lag else df_train_level
    vl_lv = df_val_lag1        if use_lag else df_val_level
    tr_df = df_train_diff_lag1 if use_lag else df_train_diff
    vl_df = df_val_diff_lag1   if use_lag else df_val_diff

    # Level model
    for emb_dim in EMB_DIMS:
        print(f'\n  Level | emb_dim={emb_dim}')
        model     = FullDemandModel(emb_dim, txt_only, use_lag, SAINT_OUT_DIM).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_LEVEL)
        scaler    = torch.amp.GradScaler('cuda')
        tr_loader = make_loader(tr_lv, True,  use_lag=use_lag)
        vl_loader = make_loader(vl_lv, False, use_lag=use_lag)
        best_loss, best_ep = float('inf'), 0
        ckpt = f'/tmp/best_{modality}_{lag_type}_{emb_dim}.pt'
        for epoch in range(1, EPOCHS_LEVEL + 1):
            tr_l = train_one_epoch(model, tr_loader, optimizer, scaler, txt_only)
            vl_l = eval_loss(model, vl_loader, txt_only)
            scheduler.step()
            if vl_l < best_loss:
                best_loss, best_ep = vl_l, epoch
                torch.save(model.state_dict(), ckpt)
            if epoch % 5 == 0 or epoch == 1:
                print(f'    Epoch {epoch:>3} | train={tr_l:.4f} | val={vl_l:.4f}')
        print(f'  Best: epoch={best_ep} | val_loss={best_loss:.4f}')
        model.load_state_dict(torch.load(ckpt))
        e_s, l_s = str(best_ep).zfill(3), f'{best_loss:.4f}'
        for split, sdf in [('train', tr_lv), ('val', vl_lv)]:
            fname = (f'{split}_pred_model-{TODAY}-embdim=None_{lag_type}_{modality}_mod4-'
                     f'epoch={e_s}-val_combined_loss={l_s}_{modality}_dim={emb_dim}_proj_emb_step=.zip')
            extract_and_save(model, sdf, txt_only, PRED_DIR + subfolder + '/' + fname, emb_dim)
        del model; torch.cuda.empty_cache()

    # Diff model
    print(f'\n  Diff | {lag_type}')
    model_d = FullDemandModel(128, txt_only, use_lag, SAINT_OUT_DIM).to(DEVICE)
    opt_d   = optim.AdamW(model_d.parameters(), lr=LR, weight_decay=1e-4)
    sch_d   = optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=EPOCHS_DIFF)
    sc_d    = torch.amp.GradScaler('cuda')
    tr_dl   = make_loader(tr_df, True,  'DELTA_SALES_RANK', 'DELTA_PRICE', use_lag)
    vl_dl   = make_loader(vl_df, False, 'DELTA_SALES_RANK', 'DELTA_PRICE', use_lag)
    best_d, best_de = float('inf'), 0
    ckpt_d = f'/tmp/best_{modality}_{lag_type}_diff.pt'
    for epoch in range(1, EPOCHS_DIFF + 1):
        tr_l = train_one_epoch(model_d, tr_dl, opt_d, sc_d, txt_only)
        vl_l = eval_loss(model_d, vl_dl, txt_only)
        sch_d.step()
        if vl_l < best_d:
            best_d, best_de = vl_l, epoch
            torch.save(model_d.state_dict(), ckpt_d)
        if epoch % 5 == 0 or epoch == 1:
            print(f'    Epoch {epoch:>3} | train={tr_l:.4f} | val={vl_l:.4f}')
    print(f'  Best: epoch={best_de} | val_loss={best_d:.4f}')
    model_d.load_state_dict(torch.load(ckpt_d))
    e_s, l_s = str(best_de).zfill(3), f'{best_d:.4f}'
    inv = 'invariant' if not use_lag else lag_type
    for split, sdf in [('train', tr_df), ('val', vl_df)]:
        fname = (f'pred_diff_shoes-diff-model-{modality}_{TODAY}-embdim=768_'
                 f'lag1-epoch={e_s}-val_combined_loss={l_s}_{modality}_{inv}_{split}.zip')
        extract_and_save(model_d, sdf, txt_only, PRED_DIR + subfolder + '/' + fname, 128)
    del model_d; torch.cuda.empty_cache()

print('\nPart 3b complete: txtimg/time_independent — next: 00_part3c_train_txt_lag1.ipynb')
